# Computer Vision with Deep Learning
Deep learning has transformed computer vision. This notebook covers the major CV tasks: image classification, object detection (YOLO, Faster R-CNN), semantic and instance segmentation (U-Net, Mask R-CNN), and Vision Transformers.

## CV Task Taxonomy
| Task | Output | Example |
|---|---|---|
| Image Classification | Single class label | "cat" |
| Object Detection | Bounding boxes + class labels | 3 cars, 1 person |
| Semantic Segmentation | Per-pixel class label | road/car/sky pixels |
| Instance Segmentation | Per-pixel label per object | each car separately |
| Depth Estimation | Per-pixel depth map | distance from camera |
| Keypoint Detection | Joint coordinates | human pose estimation |

## Image Classification
The canonical task solved by CNNs. Pre-trained ImageNet models (ResNet, EfficientNet, ViT) reach >90% top-5 accuracy.

## Object Detection Approaches
Two major paradigms:
1. **Two-Stage (Region Proposal)**: Faster R-CNN — proposes regions of interest, then classifies and refines bounding boxes
2. **Single-Stage (Dense Prediction)**: YOLO, SSD — directly predict boxes + classes from a feature grid in one pass

## YOLO (You Only Look Once)
- Divides the image into an S×S grid
- Each cell predicts B bounding boxes (x, y, w, h, confidence) + C class probabilities
- Extremely fast (single forward pass); ideal for real-time detection
- YOLOv8/YOLOv9 are current state-of-the-art speed-accuracy tradeoffs

## Faster R-CNN
- **Backbone CNN** extracts feature maps
- **Region Proposal Network (RPN)** proposes candidate bounding boxes (anchors)
- **RoI Pooling** extracts fixed-size features per proposal
- **Classification head** classifies proposals + refines box coordinates
- Slower than YOLO but typically higher accuracy on standard benchmarks

## Semantic Segmentation: U-Net
Encoder-decoder with skip connections:
- **Encoder**: Contracting path (conv + maxpool) — builds rich feature representations, shrinks spatial dims
- **Decoder**: Expanding path (transposed convolutions) — restores spatial resolution
- **Skip Connections**: Concatenate encoder feature maps to decoder layers — preserves fine-grained details

U-Net dominates medical image segmentation.

## Instance Segmentation: Mask R-CNN
Extends Faster R-CNN with a parallel mask prediction branch:
- Detection branch: class label + bounding box
- Mask branch: Binary segmentation mask for each detected instance
- Uses RoI Align (bilinear interpolation instead of quantized pooling)

In [1]:
# U-Net style encoder-decoder block in Keras
from tensorflow.keras import layers, models

def unet_block(x, filters, up=False, skip=None):
    if up:
        x = layers.UpSampling2D(2)(x)
        if skip is not None:
            x = layers.Concatenate()([x, skip])
    x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
    x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
    return x

# Minimal U-Net
inp = layers.Input(shape=(256, 256, 1))
e1 = unet_block(inp, 64)
e2 = unet_block(layers.MaxPooling2D(2)(e1), 128)
e3 = unet_block(layers.MaxPooling2D(2)(e2), 256)
bottleneck = unet_block(layers.MaxPooling2D(2)(e3), 512)
d1 = unet_block(bottleneck, 256, up=True, skip=e3)
d2 = unet_block(d1, 128, up=True, skip=e2)
d3 = unet_block(d2, 64, up=True, skip=e1)
out = layers.Conv2D(1, 1, activation='sigmoid')(d3)  # binary segmentation

unet = models.Model(inp, out, name="MiniUNet")
unet.summary()

Model: "MiniUNet"

┏━━━━━━┳━━━━━┳━━┳━━━━━━┓
┃ Lay… ┃ Ou… ┃  ┃ Con… ┃
┃ (ty… ┃ Sh… ┃  ┃ to   ┃
┡━━━━━━╇━━━━━╇━━╇━━━━━━┩
│ inp… │ (N… │  │ -    │
│ (In… │ 25… │  │      │
│      │ 25… │  │      │
│      │ 1)  │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ inp… │
│ (Co… │ 25… │  │      │
│      │ 25… │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 25… │  │      │
│      │ 25… │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ max… │ (N… │  │ con… │
│ (Ma… │ 12… │  │      │
│      │ 12… │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ max… │
│ (Co… │ 12… │  │      │
│      │ 12… │  │      │
│      │ 12… │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 12… │  │      │
│      │ 12… │  │      │
│      │ 12… │  │      │
├──────┼─────┼──┼──────┤
│ max… │ (N… │  │ con… │
│ (Ma… │ 64, │  │      │
│      │ 64, │  │      │
│      │ 12… │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ max… │
│ (Co… │ 64, │  │      │
│      │ 64, │  │      │
│      │ 25… │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 64, │  │      │
│      │ 64, │  │      │
│      │ 25… │  │      │
├──────┼─────┼──┼──────┤
│ max… │ (N… │  │ con… │
│ (Ma… │ 32, │  │      │
│      │ 32, │  │      │
│      │ 25… │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ max… │
│ (Co… │ 32, │  │      │
│      │ 32, │  │      │
│      │ 51… │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 32, │  │      │
│      │ 32, │  │      │
│      │ 51… │  │      │
├──────┼─────┼──┼──────┤
│ up_… │ (N… │  │ con… │
│ (Up… │ 64, │  │      │
│      │ 64, │  │      │
│      │ 51… │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ up_… │
│ (Co… │ 64, │  │ con… │
│      │ 64, │  │      │
│      │ 76… │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 64, │  │      │
│      │ 64, │  │      │
│      │ 25… │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 64, │  │      │
│      │ 64, │  │      │
│      │ 25… │  │      │
├──────┼─────┼──┼──────┤
│ up_… │ (N… │  │ con… │
│ (Up… │ 12… │  │      │
│      │ 12… │  │      │
│      │ 25… │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ up_… │
│ (Co… │ 12… │  │ con… │
│      │ 12… │  │      │
│      │ 38… │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 12… │  │      │
│      │ 12… │  │      │
│      │ 12… │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 12… │  │      │
│      │ 12… │  │      │
│      │ 12… │  │      │
├──────┼─────┼──┼──────┤
│ up_… │ (N… │  │ con… │
│ (Up… │ 25… │  │      │
│      │ 25… │  │      │
│      │ 12… │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ up_… │
│ (Co… │ 25… │  │ con… │
│      │ 25… │  │      │
│      │ 19… │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 25… │  │      │
│      │ 25… │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 25… │  │      │
│      │ 25… │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 25… │  │      │
│      │ 25… │  │      │
│      │ 1)  │  │      │
└──────┴─────┴──┴──────┘

 Total params: 7,781,761 (29.69 MB)

 Trainable params: 7,781,761 (29.69 MB)

 Non-trainable params: 0 (0.00 B)

# Conclusions and Key Takeaways
- Deep learning has achieved superhuman performance on most classical CV benchmarks.
- **YOLO-family** models dominate real-time detection; **Faster R-CNN** variants dominate accuracy-critical deployments.
- **U-Net** is the gold standard for medical and satellite image segmentation.
- **Mask R-CNN** is the go-to for instance segmentation.
- **Vision Transformers (ViT, DINO, SAM)** are increasingly challenging CNN dominance, especially at large scale.

# Pros and Cons
**Pros:**
- Pre-trained CNN and Transformer backbones provide excellent transfer learning for CV tasks
- End-to-end differentiable pipelines eliminate hand-crafted feature engineering
- Modern models are approaching real-time performance even for complex tasks like segmentation

**Cons:**
- Require thousands to millions of labeled training examples
- Computationally expensive training and inference
- Object detectors are sensitive to distribution shift and adversarial examples

# 15 Interview Questions and Answers

1. **What is the difference between object detection and semantic segmentation?**
   *Answer*: Detection outputs bounding boxes with class labels. Semantic segmentation outputs a per-pixel class label across the full image, with no distinction between individual object instances.

2. **How does YOLO perform detection in a single pass?**
   *Answer*: The image is divided into an S×S grid. Each cell predicts B bounding boxes with confidence scores and C class probabilities. All predictions are made simultaneously in a single forward pass through the backbone CNN.

3. **What is Non-Maximum Suppression (NMS)?**
   *Answer*: Post-processing step that removes duplicate bounding box detections. It iteratively selects the box with the highest confidence, suppresses all overlapping boxes with IoU > threshold, and repeats.

4. **What is IoU (Intersection over Union)?**
   *Answer*: The ratio of the area of intersection of two bounding boxes to the area of their union. Used to evaluate detection quality and as threshold in NMS. IoU=1 means perfect overlap.

5. **What is a Region Proposal Network (RPN) in Faster R-CNN?**
   *Answer*: A small convolutional network that slides over the feature map and proposes candidate object bounding boxes (anchors at multiple scales and aspect ratios). It outputs objectness scores and box offsets.

6. **What are anchor boxes?**
   *Answer*: Pre-defined reference boxes of various scales and aspect ratios placed at each grid cell. The model predicts offsets from anchors rather than absolute box coordinates, making learning more stable.

7. **What is the difference between RoI Pooling and RoI Align?**
   *Answer*: RoI Pooling quantizes the continuous RoI coordinates to discrete grid cells, introducing misalignment. RoI Align uses bilinear interpolation at exact floating-point positions, preserving spatial accuracy — critical for masks in Mask R-CNN.

8. **What is the U-Net architecture known for?**
   *Answer*: An encoder-decoder architecture with skip connections that concatenate encoder feature maps to the corresponding decoder layers. The skip connections preserve fine spatial details lost during downsampling, enabling precise pixel-level predictions.

9. **What makes Mask R-CNN an instance segmentation model?**
   *Answer*: It adds a parallel mask prediction head to Faster R-CNN that outputs a binary pixel mask for each separately detected bounding box, distinguishing individual object instances.

10. **What is DINO?**
    *Answer*: Self-supervised training of Vision Transformers using a self-distillation strategy. The model learns powerful visual representations (object segmentation, depth) without any labels.

11. **What is SAM (Segment Anything Model)?**
    *Answer*: A foundation model from Meta that can segment any object in an image given a point, box, or text prompt. Pre-trained on 1 billion masks; generalizes zero-shot to new domains.

12. **What is mAP (mean Average Precision)?**
    *Answer*: The primary metric for object detection. Average Precision (AP) is computed per class as the area under the Precision-Recall curve at a specific IoU threshold. mAP averages AP across all classes (and optionally across IoU thresholds).

13. **What is FPN (Feature Pyramid Network)?**
    *Answer*: A multi-scale feature representation built by combining low-resolution, semantically strong features from deep layers with high-resolution, spatially precise features from shallow layers via lateral connections and top-down pathway. Improves detection of small objects.

14. **What is Panoptic Segmentation?**
    *Answer*: Combines semantic segmentation (every pixel labeled with a class) with instance segmentation (each countable thing instance separately labeled). It provides a full scene understanding output.

15. **Why do object detectors need Data Augmentation strategies like Mosaic?**
    *Answer*: Mosaic (used in YOLOv4+) combines 4 training images into one, giving the model exposure to multiple contexts, scales, and partial occlusions simultaneously per forward pass, greatly enriching the training signal.
